In [1]:
import pandas as pd
import numpy as np
import pickle
import difflib
from pathlib import Path
from scipy import sparse
from sklearn.neighbors import NearestNeighbors

# ============================================
# Load artifacts
# ============================================
DATA_DIR = Path("../../Data/New")

df = pd.read_csv(DATA_DIR / "Dataset_Final.csv")

with open(DATA_DIR / "tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

tfidf_matrix = sparse.load_npz(DATA_DIR / "tfidf_matrix.npz")

assert df.shape[0] == tfidf_matrix.shape[0], "Row count mismatch between dataset and TF-IDF matrix!"
print("Dataset shape:", df.shape)
print("TF-IDF matrix shape:", tfidf_matrix.shape)

Dataset shape: (367703, 8)
TF-IDF matrix shape: (367703, 50000)


In [2]:
# ============================================
# Train and save NearestNeighbors model
# ============================================
nn_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=50
)
nn_model.fit(tfidf_matrix)

with open(DATA_DIR / "nearest_neighbors.pkl", "wb") as f:
    pickle.dump(nn_model, f)

print("NearestNeighbors model trained and saved.")

NearestNeighbors model trained and saved.


In [3]:
# ============================================
# Title -> row index lookup (dedup, keep first)
# ============================================
title_to_index = pd.Series(df.index, index=df['title'])
title_to_index = title_to_index[~title_to_index.index.duplicated(keep='first')]

def recommend(movie_title, top_n=10):
    """
    Returns top_n movies similar to `movie_title`, sorted by similarity_score desc.
    Returns None and prints a message (with close-match suggestions) if not found.
    """
    if movie_title not in title_to_index:
        close = difflib.get_close_matches(movie_title, title_to_index.index, n=5, cutoff=0.6)
        msg = f"'{movie_title}' not found in dataset."
        if close:
            msg += f" Did you mean: {close}?"
        print(msg)
        return None

    idx = title_to_index[movie_title]

    # queries the model's default 50 neighbors, including the movie itself
    distances, indices = nn_model.kneighbors(tfidf_matrix[idx])
    distances, indices = distances.flatten(), indices.flatten()

    results = []
    for dist, i in zip(distances, indices):
        if i == idx:
            continue  # exclude the movie itself
        results.append({
            'title': df.iloc[i]['title'],
            'similarity_score': round(1 - dist, 4),
            'vote_average': df.iloc[i]['vote_average'],
            'release_date': df.iloc[i]['release_date'],
            'poster_path': df.iloc[i]['poster_path']
        })

    rec_df = (pd.DataFrame(results)
              .sort_values('similarity_score', ascending=False)
              .head(top_n)
              .reset_index(drop=True))
    return rec_df

# Test
recommend("Interstellar", top_n=10)

,title,similarity_score,vote_average,release_date,poster_path
0,The Martian,0.2932,8.0,2015,https://m.media-amazon.com/images/M/MV5BMTc2MT...
1,Leera the Soulmate,0.2906,6.3,(2018),NaN
2,Battle Los Angeles,0.2724,NaN,2011,NaN
3,Roar: Tigers of the Sundarbans,0.2713,5.6,(2014),NaN
4,Spider-Man: Far from Home,0.2545,NaN,2019,NaN
5,To an Asteroid and Back,0.2537,0.0,2024-09-24,/k0evZx3erCK7ZuUran3lt5KjaFf.jpg
6,Superman,0.2528,4.5,1978,/d7px1FQxW4tngdACVRsCSaZq0Xl.jpg
7,Home,0.2512,8.0,2015,/cGsM2svqprMTw1FKtuG4oYUZ6ur.jpg
8,E.T. the Extra-Terrestrial,0.2479,7.8,1982,https://m.media-amazon.com/images/M/MV5BMTQ2OD...
9,Arrival,0.2457,7.9,2016,https://m.media-amazon.com/images/M/MV5BMTExMz...
